In [15]:
# ! pip install scikit-surprise

In [5]:
import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

In [6]:
fruit_ratings = pd.read_csv('fruit_ratings.csv')

In [7]:

# Surprise needs a 'long-format' DataFrame: user, item, rating

reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(fruit_ratings[['User', 'Fruit', 'Rating']], reader)
data

In [8]:

trainset, testset = train_test_split(data, test_size=0.2, random_state=42)


In [9]:

algo = SVD(n_factors=2, random_state=42)  # n_factors = latent features
algo.fit(trainset)


In [6]:

predictions = algo.test(testset)
print("RMSE:", accuracy.rmse(predictions))


RMSE: 1.6529
RMSE: 1.6528879165247474


In [10]:
# --- Make Recommendations for a new user ---
new_user_id = 'new_user'

# Suppose new user has rated Lemon=2, Mango=5
new_user_ratings = {
    'Lemon': 2,
    'Mango': 5
}


In [11]:

# Add to trainset temporarily
for fruit, rating in new_user_ratings.items():
    trainset.build_testset()  # we won't retrain, just predict


In [12]:

# Predict for all fruits
all_fruits = fruit_ratings['Fruit'].unique()
preds = []
for fruit in all_fruits:
    if fruit not in new_user_ratings:
        pred = algo.predict(new_user_id, fruit, r_ui=None)
        preds.append((fruit, pred.est))



In [13]:
# Sort recommendations
preds.sort(key=lambda x: x[1], reverse=True)


In [14]:
print("Top Recommendations:")
for fruit, rating in preds:
    print(f"{fruit}: {rating:.2f}")


Top Recommendations:
Pineapple: 3.59
Peach: 3.50
Lime: 3.23
Banana: 3.10
